In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_16_try_0.pickle

In [ ]:
%%PandasProfile
### cell 17 ###

# Optimized code

def count_percent(df, col, mapping=None):
    """
    Compute value percentages for a column, applying an optional replacement mapping.
    Returns a Series of percentages rounded to 1 decimal.
    """
    s = df[col]
    if mapping:
        s = s.replace(mapping)
    counts = s.value_counts(dropna=False)
    total = s.count()
    return (counts * 100 / total).round(1)


def combine_degrees(question, x_axis, include_2017=False):
    """
    For each year’s responses DataFrame, compute percentage distribution of `question`,
    normalize mis-encoded degree labels, and return a concatenated DataFrame with columns
    ['percentage', 'year', x_axis].
    """
    mapping = {
        'Masterâ\x80\x99s degree': "Master's degree",
        'Bachelorâ\x80\x99s degree': "Bachelor's degree"
    }
    # collect (year, DataFrame) in desired order
    year_sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        year_sources.insert(0, ('2017', responses_df_2017))

    # build a single DataFrame with a 'year' column and the question responses
    df_list = []
    for year, df_src in year_sources:
        df_work = df_src if year != '2017' else df_src.rename(columns={'FormalEducation': question})
        df_list.append(pd.DataFrame({
            'year': year,
            question: df_work[question]
        }))
    df_all = pd.concat(df_list, ignore_index=True)

    # normalize mis-encoded labels
    df_all[question] = df_all[question].replace(mapping)

    # count per (year, response) including NaNs, and total non-null per year
    counts = df_all.groupby(['year', question], dropna=False).size()
    totals = df_all.groupby('year')[question].count()

    # compute percentages
    pct = (counts.div(totals, level='year') * 100).round(1)

    # format result and reorder columns
    pct = pct.rename_axis(['year', x_axis]).reset_index(name='percentage')
    return pct[['percentage', 'year', x_axis]]

# Parameters
question_of_interest_cell29 = (
    'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
)
title_for_x_axis_cell29 = ''

# Combine 2017–2022 data
degree_df_combined = combine_degrees(
    question_of_interest_cell29,
    title_for_x_axis_cell29,
    include_2017=True
)

# Re-categorize into subset plus 'Other'
subset_of_degrees = ["Bachelor's degree", "Master's degree", 'Doctoral degree']
degree_df_combined_cell29 = (
    degree_df_combined
    .assign(**{
        title_for_x_axis_cell29: lambda df: df[title_for_x_axis_cell29]
            .where(df[title_for_x_axis_cell29].isin(subset_of_degrees), 'Other')
    })
    .sort_values(by=['year', 'percentage'], ascending=True)
)

degree_df_combined_cell29.info()
